creazione struttura cartelle per YOLO

In [10]:
DATASET_DIR = "/Users/francescopiocirillo/Desktop/artificial vision/SoccerNet/tracking" # il path dove si trova il dataset soccernet da convertire

DESTINATION_DIR = "/Users/francescopiocirillo/Desktop/artificial_vision_project" # il path dove mettere il dataset convertito in formato YOLO

In [ ]:
import os

os.makedirs(os.path.join(DESTINATION_DIR, 'dataset'), exist_ok=True)
DESTINATION_DIR = os.path.join(DESTINATION_DIR, 'dataset')

os.makedirs(os.path.join(DESTINATION_DIR, 'images'), exist_ok=True)
DESTINATION_IMAGES_DIR = os.path.join(DESTINATION_DIR, 'images')

os.makedirs(os.path.join(DESTINATION_DIR, 'labels'), exist_ok=True)
DESTINATION_LABELS_DIR = os.path.join(DESTINATION_DIR, 'labels')

DIRECTORIES_TO_CREATE = ["train", "val"]

for d in DIRECTORIES_TO_CREATE:
    os.makedirs(os.path.join(DESTINATION_IMAGES_DIR, d), exist_ok=True)
    os.makedirs(os.path.join(DESTINATION_LABELS_DIR, d), exist_ok=True)


In [12]:
import numpy as np

array_campioni_da_usare = np.arange(1, 750, 7)
print(array_campioni_da_usare)


[  1   8  15  22  29  36  43  50  57  64  71  78  85  92  99 106 113 120
 127 134 141 148 155 162 169 176 183 190 197 204 211 218 225 232 239 246
 253 260 267 274 281 288 295 302 309 316 323 330 337 344 351 358 365 372
 379 386 393 400 407 414 421 428 435 442 449 456 463 470 477 484 491 498
 505 512 519 526 533 540 547 554 561 568 575 582 589 596 603 610 617 624
 631 638 645 652 659 666 673 680 687 694 701 708 715 722 729 736 743]


In [13]:
print(len(array_campioni_da_usare))

107


spostamento foto da directory soccernet a directory YOLO

In [14]:
import shutil


for d in DIRECTORIES_TO_CREATE: # giriamo le varie parti del dataset, train, test e challenge
    DATASET_PART_DIR = os.path.join(DATASET_DIR, d)

    for folder_name in os.listdir(DATASET_PART_DIR): # giriamo tutte le cartelle in una parte del dataset
        folder_path = os.path.join(DATASET_PART_DIR, folder_name, "img1")
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                src_file = os.path.join(folder_path, file_name)
                if os.path.isfile(src_file):
                    numero = int(file_name.split('.')[0])
                    if numero in array_campioni_da_usare:
                        dst_file = os.path.join(DESTINATION_IMAGES_DIR, d, folder_name + "_" + file_name)
                        shutil.copy2(src_file, dst_file)


spostamento gt da directory soccernet a directory YOLO

In [15]:
IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080

def to_yolo_format(x, y, w, h):
    """
    Converte coordinate (x_top_left, y_top_left, width, height) in formato YOLO normalizzato.
    """
    x_center = x + w / 2
    y_center = y + h / 2
    return x_center / IMAGE_WIDTH, y_center / IMAGE_HEIGHT, w / IMAGE_WIDTH, h / IMAGE_HEIGHT


In [1]:
import configparser
import re

def get_ball_tracklet_id(ini_path):
    config = configparser.ConfigParser()
    config.read(ini_path)

    for key, value in config["Sequence"].items():
        # cerco righe tipo: trackletID_18 = ball;1
        if value.strip().startswith("ball;"):
            match = re.search(r"trackletid_(\d+)", key)
            if match:
                return match.group(1)

    return None


In [17]:
import shutil
import csv
import tempfile
import os
from collections import defaultdict



for d in DIRECTORIES_TO_CREATE: # giriamo le varie parti del dataset, train, test e challenge
    DATASET_PART_DIR = os.path.join(DATASET_DIR, d)

    for folder_name in os.listdir(DATASET_PART_DIR): # giriamo tutte le cartelle in una parte del dataset
        folder_path = os.path.join(DATASET_PART_DIR, folder_name, "gt")
        ini_file_path = os.path.join(DATASET_PART_DIR, folder_name, "gameinfo.ini")
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                src_file = os.path.join(folder_path, file_name)

                ###### rimozione delle righe di gt relative alla palla
                temp_file = tempfile.NamedTemporaryFile(mode='w', newline='', delete=False, encoding='utf-8')
                with open(src_file, newline='', encoding='utf-8') as f, temp_file:
                    reader = csv.reader(f, delimiter=',')
                    writer = csv.writer(temp_file, delimiter=',')
                    for row in reader:
                        if row[1] != get_ball_tracklet_id(ini_file_path):
                            writer.writerow(row)
                os.replace(temp_file.name, src_file)
                #######

                ##### creazione dei diversi file label
                # Dizionario per raggruppare le righe per prima colonna
                groups = defaultdict(list)

                with open(src_file, newline='', encoding='utf-8') as f:
                    reader = csv.reader(f, delimiter=',')
                    for row in reader:
                        row_0 = int(row[0])
                        if row_0 not in array_campioni_da_usare:
                            continue
                        # Rimuove le prime due colonne e mette 0 come nuova prima colonna
                        new_row = [0] + row[2:]
                        # Converte le colonne 3-6 (indice 1-4 in new_row) in formato YOLO
                        x, y, w, h = map(float, new_row[1:5])
                        new_row[1:5] = to_yolo_format(x, y, w, h)
                        new_row = new_row[:-4]
                        groups[row[0]].append(new_row)

                for key, rows in groups.items():
                    file_name = f"{folder_name}_{int(key):06d}.txt"
                    dst_file = os.path.join(DESTINATION_LABELS_DIR, d, file_name)
                    with open(dst_file, 'w', newline='', encoding='utf-8') as f_out:
                        writer = csv.writer(f_out, delimiter=' ')
                        writer.writerows(rows)
                ######
